# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0369/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes


### Signal 1 — CTR vs Position

Verdict: MIXED

The CTR-versus-position pattern is observed in the March 2026 data. A substantial number of rows have a strong search position combined with low CTR. However, the bucket counts alone do not show whether these rows would consistently benefit from an action, so I treat the signal as mixed rather than confirmed.

### Signal 2 — Volume

Verdict: MIXED

The volume signal is clearly observed in the March 2026 data, with most rows having low impressions and a smaller group having medium or high impressions. However, the bucket counts alone do not show that high-volume content is more likely to be a useful quick-win opportunity, so I treat the signal as mixed rather than confirmed.

### Baseline rule

I will prioritize content that has a relatively strong search position, low CTR, and enough impressions to represent a meaningful opportunity. The score gives higher priority to pages where the combination of search position, CTR, and impressions suggests a possible CTR improvement opportunity.

This is a rule-based baseline, not a claim that the action will definitely improve performance.

### Reason code

LOW_CTR_STRONG_POSITION — the content has a relatively strong search position but a low CTR.

### Action label

CTR_REVIEW — review the search result presentation and decide whether a CTR improvement action is appropriate.

In [2]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")



README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

ctr_position = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN 'No impressions'
        WHEN gsc_avg_position <= 3 AND (gsc_clicks * 100.0 / gsc_impressions) < 1
            THEN 'Top position + low CTR'
        WHEN gsc_avg_position <= 10 AND (gsc_clicks * 100.0 / gsc_impressions) < 2
            THEN 'Page 1 + low CTR'
        WHEN gsc_avg_position <= 10
            THEN 'Page 1'
        ELSE 'Lower position'
    END AS bucket,
    COUNT(*) AS n
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY bucket
ORDER BY n DESC
""").df()

ctr_position


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,bucket,n
0,Lower position,1427577
1,Page 1 + low CTR,1427239
2,Top position + low CTR,680844
3,Page 1,75401


In [5]:
volume_check = con.sql(f"""
SELECT
    CASE
        WHEN gsc_impressions = 0 THEN 'No impressions'
        WHEN gsc_impressions < 100 THEN 'Low volume'
        WHEN gsc_impressions < 1000 THEN 'Medium volume'
        ELSE 'High volume'
    END AS bucket,
    COUNT(*) AS n
FROM {REL}
WHERE gsc_data_available IS TRUE
GROUP BY bucket
ORDER BY
    CASE bucket
        WHEN 'No impressions' THEN 1
        WHEN 'Low volume' THEN 2
        WHEN 'Medium volume' THEN 3
        WHEN 'High volume' THEN 4
    END
""").df()

volume_check

,bucket,n
0,Low volume,2972453
1,Medium volume,606189
2,High volume,32419


In [6]:
summary = con.sql(f"""
SELECT
    COUNT(*) AS n,
    ROUND(AVG(gsc_avg_position), 2) AS avg_position,
    ROUND(AVG(
        CASE
            WHEN gsc_impressions > 0
            THEN gsc_clicks * 100.0 / gsc_impressions
        END
    ), 2) AS avg_ctr,
    ROUND(AVG(gsc_impressions), 2) AS avg_impressions
FROM {REL}
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n,avg_position,avg_ctr,avg_impressions
0,3611061,15.83,0.31,77.72


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
baseline = con.sql(f"""
WITH monthly AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        SUM(gsc_clicks) * 100.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS avg_position

    FROM {REL}

    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0

    GROUP BY
        client_hash_id,
        content_hash_id
),

scored AS (
    SELECT
        *,

        -- Lower position = stronger opportunity
        CASE
            WHEN avg_position <= 3 THEN 40
            WHEN avg_position <= 5 THEN 30
            WHEN avg_position <= 10 THEN 20
            ELSE 0
        END AS position_score,

        -- Lower CTR = greater possible CTR opportunity
        CASE
            WHEN ctr < 0.25 THEN 40
            WHEN ctr < 0.50 THEN 30
            WHEN ctr < 1.00 THEN 20
            WHEN ctr < 2.00 THEN 10
            ELSE 0
        END AS ctr_score,

        -- More impressions = larger potential impact
        CASE
            WHEN impressions >= 500000 THEN 20
            WHEN impressions >= 250000 THEN 15
            WHEN impressions >= 100000 THEN 10
            WHEN impressions >= 50000 THEN 5
            ELSE 0
        END AS volume_score

    FROM monthly
)

SELECT
    client_hash_id,
    content_hash_id,
    impressions,
    clicks,
    ROUND(ctr, 3) AS ctr,
    ROUND(avg_position, 2) AS avg_position,

    position_score,
    ctr_score,
    volume_score,

    position_score + ctr_score + volume_score AS score,

    'LOW_CTR_STRONG_POSITION' AS reason_code,

    'CTR_REVIEW' AS action

FROM scored

WHERE avg_position <= 10
  AND ctr < 2

ORDER BY
    score DESC,
    impressions DESC
""").df()

baseline.head(20)

,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_score,ctr_score,volume_score,score,reason_code,action
0,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.142,2.56,40,40,10,90,LOW_CTR_STRONG_POSITION,CTR_REVIEW
1,client_e547b89c05043229,content_545bb6cc7081ded3,122905.0,287.0,0.234,2.62,40,40,10,90,LOW_CTR_STRONG_POSITION,CTR_REVIEW
2,client_e547b89c05043229,content_9ef3d7516483e665,89229.0,92.0,0.103,2.48,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
3,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,0.043,1.49,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
4,client_73cda7b4e4f265ea,content_80eb6221de550658,79766.0,175.0,0.219,2.23,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
5,client_62f4a7e64f5e0096,content_b13e95d379c78818,76121.0,151.0,0.198,1.14,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
6,client_73cda7b4e4f265ea,content_6a9c79f55413b447,73272.0,118.0,0.161,2.55,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
7,client_e547b89c05043229,content_c46df0fa61530d86,70398.0,42.0,0.060,1.56,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
8,client_62f4a7e64f5e0096,content_6b4d5458bf2143b1,67642.0,164.0,0.242,2.85,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
9,client_73cda7b4e4f265ea,content_252aa5480bb1f8d7,66698.0,75.0,0.112,2.39,40,40,5,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW


In [12]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows in ranked queue:", len(baseline))
print("CSV written to: work/outputs/baseline_action_score.csv")

Rows in ranked queue: 91723
CSV written to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

The top 20 items are mostly strong-position pages with low CTR. I treat these as review candidates rather than guaranteed problems.

The confidence is moderate because the baseline only uses three search signals. Search intent, SERP features, query type, and other factors could explain some low CTR values.

In [13]:
top20 = baseline.head(20).copy()

top20.insert(0, "rank", range(1, len(top20) + 1))

top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "score",
        "reason_code",
        "action"
    ]
]


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,score,reason_code,action
0,1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,0.142,2.56,90,LOW_CTR_STRONG_POSITION,CTR_REVIEW
1,2,client_e547b89c05043229,content_545bb6cc7081ded3,122905.0,287.0,0.234,2.62,90,LOW_CTR_STRONG_POSITION,CTR_REVIEW
2,3,client_e547b89c05043229,content_9ef3d7516483e665,89229.0,92.0,0.103,2.48,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
3,4,client_e547b89c05043229,content_306bc78dff1eb683,80821.0,35.0,0.043,1.49,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
4,5,client_73cda7b4e4f265ea,content_80eb6221de550658,79766.0,175.0,0.219,2.23,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
5,6,client_62f4a7e64f5e0096,content_b13e95d379c78818,76121.0,151.0,0.198,1.14,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
6,7,client_73cda7b4e4f265ea,content_6a9c79f55413b447,73272.0,118.0,0.161,2.55,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
7,8,client_e547b89c05043229,content_c46df0fa61530d86,70398.0,42.0,0.060,1.56,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
8,9,client_62f4a7e64f5e0096,content_6b4d5458bf2143b1,67642.0,164.0,0.242,2.85,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW
9,10,client_73cda7b4e4f265ea,content_252aa5480bb1f8d7,66698.0,75.0,0.112,2.39,85,LOW_CTR_STRONG_POSITION,CTR_REVIEW


### Top-20 individual review

1. **Rank 1 — CTR_REVIEW:** Position 2.56, CTR 0.142%, and 203,497 impressions make this a strong match for the rule. It could be wrong if the underlying search intent naturally produces few clicks.

2. **Rank 2 — CTR_REVIEW:** Position 2.62, CTR 0.234%, and 122,905 impressions indicate a strong-position, low-CTR pattern. It could be wrong if SERP features reduce clicks.

3. **Rank 3 — CTR_REVIEW:** Position 2.48, CTR 0.103%, and 89,229 impressions make this a strong review candidate. It could be wrong if the queries have naturally low click intent.

4. **Rank 4 — CTR_REVIEW:** Position 1.49, CTR 0.043%, and 80,821 impressions show very low CTR despite an excellent position. It could be wrong if the result is affected by unusual SERP behavior.

5. **Rank 5 — CTR_REVIEW:** Position 2.23, CTR 0.219%, and 79,766 impressions fit the rule well. It could be wrong if search intent explains the low CTR.

6. **Rank 6 — CTR_REVIEW:** Position 1.14, CTR 0.198%, and 76,121 impressions make this a strong candidate. It could be wrong if users do not normally click this type of result.

7. **Rank 7 — CTR_REVIEW:** Position 2.55, CTR 0.161%, and 73,272 impressions indicate a possible CTR opportunity. It could be wrong if SERP presentation is limiting clicks.

8. **Rank 8 — CTR_REVIEW:** Position 1.56, CTR 0.060%, and 70,398 impressions show a strong mismatch between position and CTR. It could be wrong because the query mix may be unusual.

9. **Rank 9 — CTR_REVIEW:** Position 2.85, CTR 0.242%, and 67,642 impressions fit the baseline rule. It could be wrong if the low CTR is normal for the relevant searches.

10. **Rank 10 — CTR_REVIEW:** Position 2.39, CTR 0.112%, and 66,698 impressions make this a reasonable review candidate. It could be wrong because search intent may limit clicks.

11. **Rank 11 — CTR_REVIEW:** Position 2.94, CTR 0.170%, and 65,995 impressions indicate a strong position with low CTR. It could be wrong if SERP features affect click behavior.

12. **Rank 12 — CTR_REVIEW:** Position 1.54, CTR 0.093%, and 65,304 impressions make this a strong signal match. It could be wrong if the underlying queries have low click intent.

13. **Rank 13 — CTR_REVIEW:** Position 2.28, CTR 0.236%, and 61,347 impressions fit the rule. It could be wrong if the observed CTR is expected for those searches.

14. **Rank 14 — CTR_REVIEW:** Position 2.26, CTR 0.030%, and 60,172 impressions show very low CTR at a strong position. It could be wrong if SERP features or query intent explain the result.

15. **Rank 15 — CTR_REVIEW:** Position 2.73, CTR 0.123%, and 58,588 impressions indicate a possible CTR opportunity. It could be wrong if users have little reason to click the result.

16. **Rank 16 — CTR_REVIEW:** Position 2.41, CTR 0.151%, and 55,451 impressions fit the baseline conditions. It could be wrong if the query mix naturally produces low CTR.

17. **Rank 17 — CTR_REVIEW:** Position 2.74, CTR 0.140%, and 54,208 impressions make this a review candidate. It could be wrong if the search results page provides answers without requiring a click.

18. **Rank 18 — CTR_REVIEW:** Position 2.89, CTR 0.243%, and 53,132 impressions fit the rule. It could be wrong if the low CTR is caused by search intent rather than content quality.

19. **Rank 19 — CTR_REVIEW:** Position 2.89, CTR 0.116%, and 51,722 impressions indicate a strong-position, low-CTR pattern. It could be wrong if the queries are not click-oriented.

20. **Rank 20 — CTR_REVIEW:** Position 2.93, CTR 0.130%, and 50,848 impressions make this a reasonable candidate for review. It could be wrong if SERP features or query intent explain the low CTR.

## 4. Weak picks + leakage check

The weakest picks are not necessarily incorrect, but the rule cannot distinguish between a genuine content opportunity and a low-CTR situation caused by search intent or SERP layout.

The baseline should therefore be treated as a review queue rather than an automatic action system.

The rule uses March 2026 Search Console observations only. I did not use future-window performance or a label-derived field. Client and content hash IDs are used only to identify the rows.

In [14]:
print("Rows in baseline:", len(baseline))
print("Minimum score:", baseline["score"].min())
print("Maximum score:", baseline["score"].max())

print("\nReason codes:")
print(baseline["reason_code"].value_counts())

print("\nActions:")
print(baseline["action"].value_counts())

print("\nLeakage check:")
print("Future-window data used: NO")
print("Label-derived fields used: NO")
print("Product flags used: NO")
print("Client/content IDs used as score inputs: NO")

Rows in baseline: 91723
Minimum score: 30
Maximum score: 90

Reason codes:
reason_code
LOW_CTR_STRONG_POSITION    91723
Name: count, dtype: int64

Actions:
action
CTR_REVIEW    91723
Name: count, dtype: int64

Leakage check:
Future-window data used: NO
Label-derived fields used: NO
Product flags used: NO
Client/content IDs used as score inputs: NO


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.